In [ ]:

# CONFIGURAÇÕES INICIAIS DAS ANÁLISES (PRESENTES EM TODOS OS SCRIPTS)
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent        # caminho desse script
ANALYTICS_DIR = SCRIPT_DIR.parent                   # pasta desse script

# PARA IMPORTAR FUNÇÕES DE EXTRAIR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv


# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 01 - Como está estruturado o mercado brasileiro de Dados? ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO ---
caminho_gold_01 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_01_estrutura_mercado")

# BUSCA CSVs GERADOS PELO SPARK NA CRIAÇÃO DA GOLD ---
arquivos_gold_01 = [
    str(arquivo)
    for arquivo in caminho_gold_01.glob("part-*.csv")
]
print("Arquivos encontrados:")
print(arquivos_gold_01)

# CARREGAR GOLD 01 ---
df_estrutura = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_01)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INÍCIO DAS ANÁLISES ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# MODELO DE TRABALHO ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_modelo_trabalho = (df_estrutura
    .filter(F.col("variavel") == "modelo_de_trabalho_atual")
    .orderBy("edicao",F.desc("pct_na_dimensao"))
)
df_modelo_trabalho.show(100, truncate=False)
print("Qtd linhas:", df_modelo_trabalho.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A dimensão de modelo de trabalho está disponível para as edições
2024-2025 e 2025-2026, totalizando oito registros.

Em cada edição são observadas quatro categorias:

- Modelo 100% remoto
- Modelo 100% presencial
- Modelo híbrido com dias fixos de trabalho presencial
- Modelo híbrido flexível

O número de respondentes varia entre os períodos:

- 2024-2025: 4.863 respondentes
- 2025-2026: 3.228 respondentes

Por esse motivo, as comparações entre as edições serão realizadas
principalmente pela participação percentual de cada modelo de trabalho,
e não pela contagem absoluta de respondentes.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# LISTA DE MODELOS DE TRABALHO EXISTENTES POR EDIÇÃO ---
(df_modelo_trabalho
    .select("edicao","valor")
    .distinct()
    .orderBy("edicao","valor")
    .show(100, truncate=False)
)

# QUANTIDADE DE CATEGORIAS POR EDIÇÃO ---
(df_modelo_trabalho
    .groupBy("edicao")
    .agg(F.countDistinct("valor").alias("qtd_modelos"))
    .orderBy("edicao")
    .show(truncate=False)
)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A validação confirma a existência de quatro categorias de modelo
de trabalho em cada uma das duas edições analisadas.

Além da mesma quantidade, as categorias apresentam a mesma
nomenclatura nos dois períodos.

Dessa forma, não foi necessária harmonização de taxonomia e os
modelos de trabalho podem ser comparados diretamente entre
2024-2025 e 2025-2026.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# COMPARATIVO DOS MODELOS DE TRABALHO ENTRE AS EDIÇÕES ---
comparativo_modelo_trabalho = (df_modelo_trabalho
    .groupBy("valor")
    .pivot("edicao",["2024-2025", "2025-2026"])
    .agg(F.first("pct_na_dimensao"))
)
comparativo_modelo_trabalho.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O modelo 100% remoto permanece como a modalidade com maior
participação nas duas edições analisadas.

Em 2024-2025, o trabalho 100% remoto representa 45,7% da amostra,
enquanto em 2025-2026 representa 39,7%.

No mesmo período, o modelo 100% presencial passa de 16,3% para
20,8%, e o modelo híbrido com dias fixos passa de 17,5% para 20,0%.

O modelo híbrido flexível apresenta participação relativamente
estável, passando de 20,5% para 19,5%.

Assim, apesar de o trabalho totalmente remoto continuar sendo a
categoria individual mais frequente, sua participação é menor na
edição mais recente.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# VARIAÇÃO ENTRE 2024-2025 E 2025-2026 ---
comparativo_modelo_trabalho = (comparativo_modelo_trabalho
    .withColumn("variacao_pp",F.round(F.col("2025-2026") - F.col("2024-2025"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_modelo_trabalho.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A comparação entre as duas edições evidencia uma mudança na
composição da amostra em relação aos modelos de trabalho.

O modelo 100% presencial apresenta o maior aumento de participação,
passando de 16,3% para 20,8%, uma variação de +4,5 pontos percentuais.

O modelo híbrido com dias fixos também aumenta sua participação,
de 17,5% para 20,0%, uma variação de +2,5 p.p.

O modelo híbrido flexível apresenta pequena redução de -1,0 p.p.,
passando de 20,5% para 19,5%.

A maior redução ocorre no modelo 100% remoto, que passa de 45,7%
para 39,7%, uma diferença de -6,0 pontos percentuais.

Os resultados indicam uma mudança na composição dos respondentes
em direção a maior participação de modalidades que possuem algum
componente presencial.

PONTO DE ATENÇÃO:
A análise descreve os respondentes das pesquisas e não permite
afirmar isoladamente que todas as empresas do mercado brasileiro
estão reduzindo ou ampliando determinado modelo de trabalho.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ESTRUTURA ATUAL DO MODELO DE TRABALHO - 2025-2026 ---
modelo_trabalho_atual = (df_modelo_trabalho
    .filter(F.col("edicao") == "2025-2026")
    .select("valor","contagem","pct_na_dimensao")
    .orderBy(F.desc("pct_na_dimensao"))
)
modelo_trabalho_atual.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição mais recente, 2025-2026, o modelo 100% remoto continua
sendo a categoria individual mais representativa, com 39,7%
dos respondentes.

Na sequência aparecem:

- Modelo 100% presencial: 20,8%
- Modelo híbrido com dias fixos: 20,0%
- Modelo híbrido flexível: 19,5%

Apesar da liderança individual do modelo remoto, os outros três
modelos apresentam participações bastante próximas entre si,
variando entre 19,5% e 20,8%.

A estrutura atual indica, portanto, uma distribuição mais equilibrada
entre as modalidades que possuem algum nível de presença física,
enquanto o remoto permanece isoladamente como a maior categoria.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CONSOLIDAÇÃO: REMOTO X MODELOS COM COMPONENTE PRESENCIAL ---
modelo_consolidado = (df_modelo_trabalho
    .withColumn("grupo_modelo",F.when(F.col("valor") == "Modelo 100% remoto","100% remoto").otherwise("Com componente presencial"))
    .groupBy("edicao","grupo_modelo")
    .agg(F.round(F.sum("pct_na_dimensao"),2).alias("pct_na_dimensao"))
    .orderBy("edicao",F.desc("pct_na_dimensao"))
)
modelo_consolidado.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A consolidação dos modelos em dois grandes grupos evidencia com
mais clareza a mudança observada entre as edições.

Em 2024-2025:

- 100% remoto: 45,7%
- Modelos com componente presencial: 54,3%

Em 2025-2026:

- 100% remoto: 39,7%
- Modelos com componente presencial: 60,3%

Assim, a participação conjunta dos modelos que envolvem algum nível
de trabalho presencial aumenta de 54,3% para 60,3%, uma variação
de +6,0 pontos percentuais.

No sentido oposto, a participação do modelo 100% remoto diminui
6,0 pontos percentuais.

Esse resultado reforça que, na composição da amostra mais recente,
seis em cada dez respondentes trabalham em modelos que possuem
algum componente presencial.

PONTO DE ATENÇÃO:
O agrupamento "Com componente presencial" reúne situações diferentes:
100% presencial, híbrido com dias fixos e híbrido flexível.
Portanto, o indicador deve ser utilizado para avaliar a presença
ou ausência de componente presencial, e não como uma categoria
única de modelo de trabalho.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# MODELO DE TRABALHO ATUAL.CSV ---
exportar_csv(modelo_trabalho_atual,OUTPUT_DIR,"modelo_trabalho_atual.csv")

# HISTÓRICO DO MODELO DE TRABALHO.CSV ---
exportar_csv(comparativo_modelo_trabalho,OUTPUT_DIR,"modelo_trabalho_historico.csv")

# MODELO DE TRABALHO CONSOLIDADO.CSV ---
exportar_csv(modelo_consolidado,OUTPUT_DIR,"modelo_trabalho_consolidado.csv")